# IPN 13 lớp — so sánh V4 kg_logits với neural-only

Notebook này dựng và đánh giá **learned V4**, không phải rule matcher. Nhánh kg_logits là một learned template-guided head: nó học từ **neural memory** của clip và **graph memory** của instance graph, rồi query 13 template graph. Vì vậy kg_logits không phải điểm graph-only hay kết quả của scripts/eval_ipn_template_matching.py; script matcher thuần luật không được gọi ở đây.

Train chỉ dùng train/validation và chọn best.pt theo macro_f1 trên validation. Test được bật bởi RUN_LOCKED_TEST=True (đặt False nếu chỉ muốn train/validation); khi gate bật, notebook chạy V4 đúng một lần (xuất đồng thời neural/KG/final), chạy neural-only độc lập đúng một lần, rồi lưu accuracy, macro-F1, per-class và confusion matrices.

**Lưu ý velocity:** neural C có velocity (5 đặc trưng/điểm) còn V4 chỉ nhận 3 đặc trưng. Notebook tự train checkpoint no-motion cùng cấu hình C để khởi tạo V4 (bước 4b); C vẫn là neural-only baseline độc lập.

Artifact train/val c46 chỉ có 2.443/589 graph; notebook chỉ precompute phần test còn thiếu (1.101 graph). EVAL_ONLY=True cho phép dùng một checkpoint V4 IPN 13 lớp đã có, nhưng vẫn bắt buộc tìm và kiểm tra test graph trước khi đánh giá.

In [ ]:
from pathlib import Path
import csv
import datetime as dt
import hashlib
import json
import shutil
import subprocess
import sys
import uuid
import zipfile

from google.colab import drive
drive.mount('/content/drive')

# Chỉ sửa các biến cấu hình ở đây. Không ghi raw keypoint/video vào artifact cuối.
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
# Artifact train/val c46: thư mục trên Drive (runs/ipn_symbolic_v2_post_c/train_val/c46e0e469d2cf3b6) hoặc file .zip tương đương.
TRAIN_VAL_ARTIFACT_ZIP = DRIVE_DATA_ROOT / 'runs' / 'ipn_symbolic_v2_post_c' / 'train_val' / 'c46e0e469d2cf3b6'
# Ablation C (masked_mean, light dropout); dùng run hoàn chỉnh, không dùng thư mục *_incomplete_* hay bigru_*.
IPN_NEURAL_BASELINE_CHECKPOINT = DRIVE_DATA_ROOT / 'outputs' / 'neural_ablation_C_masked_mean_light_dropout' / 'transformer_seed42' / 'best.pt'
IPN_V4_CHECKPOINT = None  # Path V4 13 lớp nếu chạy EVAL_ONLY.
# V4 adapter không hỗ trợ velocity (input 52*3) nhưng neural C được train CÓ velocity (52*5).
# Notebook huấn luyện một checkpoint neural cùng cấu hình C nhưng --disable-motion-features để khởi tạo V4;
# C vẫn được đánh giá độc lập làm neural-only baseline mạnh nhất.
IPN_V4_INIT_CHECKPOINT = DRIVE_DATA_ROOT / 'outputs' / 'ipn13_v4_init_no_motion' / 'transformer_seed42' / 'best.pt'
EVAL_ONLY=False
RUN_LOCKED_TEST=True  # Full test cuối: V4 (neural/KG/final) + neural-only, mỗi evaluator đúng một lần.
IPN_NEURAL_CACHE_ROOT = None  # Tùy chọn; mặc định dò cache T<num_frames> trên Drive.

REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
REPO_COMMIT = 'e817f16e0d504e95064b889288f06231bcf692ab'
EXPECTED_COUNTS = {'train': 2443, 'val': 589, 'test': 1101}
EXPECTED_NUM_CLASSES = 13
EXPECTED_TEMPLATE_IDS = set(range(EXPECTED_NUM_CLASSES))
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:10]
WORK_ROOT = Path('/content') / f'ipn13_kg_logits_{RUN_ID}'
REPO_DIR = WORK_ROOT / 'repo'
ARTIFACT_EXTRACT_DIR = WORK_ROOT / 'train_val_artifact'
MANIFEST_DIR = WORK_ROOT / 'manifests'
V4_OUTPUT_DIR = WORK_ROOT / 'v4_train'
EVAL_OUTPUT_DIR = WORK_ROOT / 'evaluation'
LOCAL_RESULTS_DIR = WORK_ROOT / 'results'
DRIVE_RUN_ROOT = DRIVE_DATA_ROOT / 'runs' / 'ipn13_kg_logits'

assert EXPECTED_COUNTS == {'train': 2443, 'val': 589, 'test': 1101}
assert EXPECTED_NUM_CLASSES == 13
assert not (EVAL_ONLY and IPN_V4_CHECKPOINT is None), 'EVAL_ONLY cần IPN_V4_CHECKPOINT.'
print({'run_id': RUN_ID, 'eval_only': EVAL_ONLY, 'locked_test': RUN_LOCKED_TEST})

## 1. Clone revision cố định và chạy lệnh bằng argv

Mọi lệnh external đều là list argv qua subprocess.run(..., shell=False) mặc định. Revision phải resolve đúng e817f16; không chạy branch trôi nổi.

In [ ]:
COMMANDS = []
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def run_argv(argv, cwd=None, capture_output=False):
    command = [str(value) for value in argv]
    COMMANDS.append(command)
    print('$', ' '.join(repr(value) for value in command))
    return subprocess.run(
        command,
        cwd=str(cwd) if cwd is not None else None,
        check=True,
        text=True,
        capture_output=capture_output,
    )

if not REPO_DIR.exists():
    run_argv(['git', 'clone', '--filter=blob:none', REPO_URL, REPO_DIR])
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'{REPO_DIR} tồn tại nhưng không phải Git repository.')
run_argv(['git', '-C', REPO_DIR, 'fetch', '--depth', '1', 'origin', REPO_COMMIT])
run_argv(['git', '-C', REPO_DIR, 'checkout', '--detach', REPO_COMMIT])
revision_result = run_argv(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True)
REPO_COMMIT_RESOLVED = revision_result.stdout.strip()
assert REPO_COMMIT_RESOLVED == REPO_COMMIT, (REPO_COMMIT_RESOLVED, REPO_COMMIT)

REQUIREMENTS_PATH = REPO_DIR / 'requirements.txt'
if REQUIREMENTS_PATH.is_file():
    run_argv([sys.executable, '-m', 'pip', 'install', '-q', '-r', REQUIREMENTS_PATH])

sys.path.insert(0, str(REPO_DIR / 'final' / 'code'))
sys.path.insert(0, str(REPO_DIR))
PRECOMPUTE_SCRIPT = REPO_DIR / 'final' / 'code' / 'src_neurosymbolic' / 'precompute_instance_graphs_phase_v2.py'
V4_TRAIN_SCRIPT = REPO_DIR / 'final' / 'code' / 'src_neurosymbolic' / 'train_cross_attention_v4.py'
V4_TEST_SCRIPT = REPO_DIR / 'final' / 'code' / 'src_neurosymbolic' / 'test_cross_attention_v2.py'
NEURAL_TEST_SCRIPT = REPO_DIR / 'final' / 'code' / 'neural_only_experiments' / 'test_neural_only.py'
for required_script in (PRECOMPUTE_SCRIPT, V4_TRAIN_SCRIPT, V4_TEST_SCRIPT, NEURAL_TEST_SCRIPT):
    assert required_script.is_file(), required_script
print('Repository revision:', REPO_COMMIT_RESOLVED)

## 2. Giải nén và kiểm tra artifact c46

Artifact train/val là provenance đầu vào, không phải checkpoint V4. Các marker completed_run.json, run_config.json, code_revision.txt, số graph và 13 template đều là hard gate.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def find_artifact_zip(configured):
    configured = Path(configured)
    if configured.exists():
        return configured.resolve()
    stem = configured.name.removesuffix('.zip')
    candidates = []
    if DRIVE_DATA_ROOT.is_dir():
        candidates = [path for path in DRIVE_DATA_ROOT.rglob(stem) if path.is_dir() and (path / 'completed_run.json').is_file()]
        candidates += sorted(DRIVE_DATA_ROOT.rglob(stem + '*.zip'))
    if len(candidates) != 1:
        raise FileNotFoundError(f'Không xác định được artifact train/val {stem}: {candidates}')
    return candidates[0].resolve()

def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    if destination.exists() and any(destination.iterdir()):
        raise FileExistsError(f'Không overwrite thư mục giải nén: {destination}')
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        names = [item.filename for item in archive.infolist()]
        for name in names:
            member = Path(name)
            if member.is_absolute() or '..' in member.parts:
                raise ValueError(f'ZIP member unsafe: {name}')
            target = (destination / name).resolve()
            try:
                target.relative_to(destination)
            except ValueError as exc:
                raise ValueError(f'ZIP member escapes destination: {name}') from exc
        archive.extractall(destination)
        roots = {Path(name).parts[0] for name in names if Path(name).parts}
    if len(roots) != 1:
        raise ValueError(f'Expected one artifact root, got {roots}')
    return destination / next(iter(roots))

TRAIN_VAL_ARTIFACT_ZIP = find_artifact_zip(TRAIN_VAL_ARTIFACT_ZIP)
if TRAIN_VAL_ARTIFACT_ZIP.is_dir():
    # Copy về local: graph test sẽ được ghi thêm vào đây và đọc Drive từng file rất chậm.
    ARTIFACT_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    ARTIFACT_ROOT = ARTIFACT_EXTRACT_DIR / TRAIN_VAL_ARTIFACT_ZIP.name
    if ARTIFACT_ROOT.exists():
        raise FileExistsError(f'Không overwrite thư mục artifact: {ARTIFACT_ROOT}')
    shutil.copytree(TRAIN_VAL_ARTIFACT_ZIP, ARTIFACT_ROOT)
else:
    assert TRAIN_VAL_ARTIFACT_ZIP.suffix.lower() == '.zip'
    ARTIFACT_ROOT = safe_extract(TRAIN_VAL_ARTIFACT_ZIP, ARTIFACT_EXTRACT_DIR)
COMPLETED_RUN_PATH = ARTIFACT_ROOT / 'completed_run.json'
RUN_CONFIG_PATH = ARTIFACT_ROOT / 'run_config.json'
CODE_REVISION_PATH = ARTIFACT_ROOT / 'code_revision.txt'
for marker in (COMPLETED_RUN_PATH, RUN_CONFIG_PATH, CODE_REVISION_PATH):
    assert marker.is_file(), marker
completed_run = json.loads(COMPLETED_RUN_PATH.read_text(encoding='utf-8'))
run_config = json.loads(RUN_CONFIG_PATH.read_text(encoding='utf-8'))
code_revision = CODE_REVISION_PATH.read_text(encoding='utf-8').strip()
assert completed_run.get('status') == 'complete'
assert completed_run.get('run_mode') == 'train_val'
assert completed_run.get('graph_counts') == {'train': 2443, 'val': 589}
assert code_revision == REPO_COMMIT_RESOLVED == REPO_COMMIT
assert run_config.get('commit') == REPO_COMMIT
assert run_config.get('active_splits') == ['train', 'val']
assert run_config.get('expected_counts') == EXPECTED_COUNTS
assert sha256_file(RUN_CONFIG_PATH) == completed_run['run_config_sha256']
payload_sha256 = completed_run.get('payload_sha256', {})
if payload_sha256:
    assert sha256_file(CODE_REVISION_PATH) == payload_sha256.get('code_revision.txt')
    template_yaml_path = ARTIFACT_ROOT / 'gesture_templates.yaml'
    if template_yaml_path.is_file() and payload_sha256.get('gesture_templates.yaml'):
        assert sha256_file(template_yaml_path) == payload_sha256['gesture_templates.yaml']

GRAPH_ROOT = ARTIFACT_ROOT / 'data' / 'instance_graphs'
TEMPLATE_ROOT = ARTIFACT_ROOT / 'data' / 'template_graphs'
def graph_files(split):
    return sorted((GRAPH_ROOT / split).rglob('*.json'))
assert len(graph_files('train')) == EXPECTED_COUNTS['train']
assert len(graph_files('val')) == EXPECTED_COUNTS['val']
assert len(graph_files('test')) == 0, 'Artifact train_val không được chứa test graph.'
ORIGINAL_MANIFESTS = ARTIFACT_ROOT / 'original_manifests'
assert (ORIGINAL_MANIFESTS / 'train.txt').is_file() and (ORIGINAL_MANIFESTS / 'val.txt').is_file()
assert len((ORIGINAL_MANIFESTS / 'train.txt').read_text(encoding='utf-8').splitlines()) == EXPECTED_COUNTS['train']
assert len((ORIGINAL_MANIFESTS / 'val.txt').read_text(encoding='utf-8').splitlines()) == EXPECTED_COUNTS['val']
template_jsons = sorted(path for path in TEMPLATE_ROOT.glob('*.json') if path.name != 'index.json')
template_index = json.loads((TEMPLATE_ROOT / 'index.json').read_text(encoding='utf-8'))
assert len(template_jsons) == EXPECTED_NUM_CLASSES
assert template_index.get('template_count') == EXPECTED_NUM_CLASSES
template_ids = {int(json.loads(path.read_text(encoding='utf-8'))['label_id']) for path in template_jsons}
assert template_ids == EXPECTED_TEMPLATE_IDS
assert not (ARTIFACT_ROOT / 'best.pt').exists(), 'Artifact train_val không được chứa V4 checkpoint.'
print({'artifact': str(TRAIN_VAL_ARTIFACT_ZIP), 'train_graphs': len(graph_files('train')), 'val_graphs': len(graph_files('val')), 'templates': len(template_jsons)})

## 3. Manifest và keypoint trên Drive

Manifest được tìm trong Drive, kiểm tra đủ 13 ID liên tục và được viết lại thành manifest local với đường dẫn keypoint đã resolve. Việc viết lại chỉ là metadata nhỏ; raw keypoint vẫn ở Drive.

In [ ]:
from src_neurosymbolic.models.keypoint_dataset import read_split_file

def manifest_candidates(name):
    preferred = [
        DRIVE_DATA_ROOT / name,
        DRIVE_DATA_ROOT / 'data' / name,
        DRIVE_DATA_ROOT / 'manifests' / name,
        DRIVE_DATA_ROOT / 'splits' / name,
    ]
    discovered = sorted(DRIVE_DATA_ROOT.rglob(name)) if DRIVE_DATA_ROOT.is_dir() else []
    return list(dict.fromkeys(path.resolve() for path in preferred + discovered if path.is_file()))

def source_label(source, fallback_split):
    parts = source.replace(chr(92), '/').split('/')
    split_index = parts.index(fallback_split) if fallback_split in parts else -1
    if split_index >= 0 and split_index + 1 < len(parts):
        return parts[split_index + 1]
    for split_name in ('train', 'val', 'test'):
        if split_name in parts and parts.index(split_name) + 1 < len(parts):
            return parts[parts.index(split_name) + 1]
    return None

def load_drive_manifest(split):
    valid = []
    for path in manifest_candidates(f'{split}.txt'):
        try:
            rows = read_split_file(path)
        except Exception:
            continue
        if len(rows) == EXPECTED_COUNTS[split] and {label for _, label in rows} == EXPECTED_TEMPLATE_IDS:
            valid.append((path, rows))
    if not valid:
        raise FileNotFoundError(f'Không tìm thấy manifest {split}.txt đúng {EXPECTED_COUNTS[split]} dòng/13 lớp.')
    preferred = [item for item in valid if item[0].parent in {DRIVE_DATA_ROOT.resolve(), (DRIVE_DATA_ROOT / 'data').resolve()}]
    if len(preferred) == 1:
        return preferred[0]
    if len(valid) != 1:
        raise RuntimeError(f'Multiple manifest candidates for {split}: {[str(path) for path, _ in valid]}')
    return valid[0]

DRIVE_MANIFESTS = {}
MANIFEST_ROWS = {}
for split in ('train', 'val', 'test'):
    DRIVE_MANIFESTS[split], MANIFEST_ROWS[split] = load_drive_manifest(split)
    ids = {label for _, label in MANIFEST_ROWS[split]}
    assert ids == EXPECTED_TEMPLATE_IDS, (split, ids)
    assert len(MANIFEST_ROWS[split]) == EXPECTED_COUNTS[split]

def resolve_keypoint_for_source(source, split, root):
    source_text = source.replace(chr(92), '/')
    parts = [part for part in source_text.split('/') if part]
    filename = parts[-1]
    candidates = []
    if 'keypoints' in parts:
        candidates.append(root.joinpath(*parts[parts.index('keypoints') + 1:]))
    if split in parts:
        split_index = parts.index(split)
        candidates.append(root.joinpath(*parts[split_index:]))
        if split_index + 2 < len(parts):
            candidates.append(root / split / parts[split_index + 1] / Path(*parts[split_index + 2:]))
    candidates.extend([root / split / filename, root / filename])
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    label = source_label(source, split)
    if label is not None:
        hits = list((root / split / label).rglob(filename)) if (root / split / label).is_dir() else []
        if len(hits) == 1:
            return hits[0].resolve()
    raise FileNotFoundError(f'Không resolve được keypoint: {source} dưới {root}')

keypoint_roots = [
    DRIVE_DATA_ROOT,
    DRIVE_DATA_ROOT / 'keypoints',
    DRIVE_DATA_ROOT / 'data' / 'keypoints',
]
KEYPOINT_ROOT = None
for candidate_root in keypoint_roots:
    if not candidate_root.is_dir():
        continue
    try:
        for split in ('train', 'val', 'test'):
            resolve_keypoint_for_source(MANIFEST_ROWS[split][0][0], split, candidate_root)
    except (FileNotFoundError, IndexError):
        continue
    KEYPOINT_ROOT = candidate_root.resolve()
    break
if KEYPOINT_ROOT is None:
    raise FileNotFoundError('Không tìm thấy keypoint root chứa cả train/val/test trên Drive.')

def materialize_manifest(split):
    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    output = MANIFEST_DIR / f'{split}.txt'
    with output.open('w', encoding='utf-8') as handle:
        for source, label in MANIFEST_ROWS[split]:
            resolved = resolve_keypoint_for_source(source, split, KEYPOINT_ROOT)
            handle.write(f'{resolved} 0 {label}\n')
    return output

TRAIN_FILE = materialize_manifest('train')
VAL_FILE = materialize_manifest('val')
TEST_FILE = materialize_manifest('test')
MANIFEST_SHA256 = {split: sha256_file(path) for split, path in [('train', TRAIN_FILE), ('val', VAL_FILE), ('test', TEST_FILE)]}
print({'keypoint_root': str(KEYPOINT_ROOT), 'manifests': {key: str(value) for key, value in DRIVE_MANIFESTS.items()}, 'normalized_hashes': MANIFEST_SHA256})

## 4. Đọc metadata baseline trước khi dựng V4

V4 phải có cùng d_model, layers, heads, dropout, num_frames với checkpoint baseline. Không hardcode cấu hình cũ 128/3/4. Checkpoint baseline C 192/4/8 được dùng nếu input projection tương thích. V4 adapter không có motion-feature projection và dùng mean pooling khi tạo neural memory; vì vậy use_motion_features=True và attention pooling bị từ chối. masked_mean vẫn shape-compatible nhưng khác semantics pooling, nên neural-only được đánh giá riêng bằng evaluator của nó.

In [ ]:
import torch
from src_neurosymbolic.models.cross_attention_v2 import load_checkpoint_compat

def checkpoint_path(value, name):
    if value is None:
        raise ValueError(f'{name} chưa được cấu hình.')
    path = Path(value).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f'{name} không tồn tại: {path}')
    return path.resolve()

if IPN_NEURAL_BASELINE_CHECKPOINT is None:
    # Tự dò checkpoint neural-only 13 lớp tương thích V4 trên Drive; mơ hồ thì dừng và liệt kê.
    found = []
    for candidate in sorted(DRIVE_DATA_ROOT.rglob('*.pt')):
        if 'v4' in str(candidate).lower() or candidate.parent.name == 'payload':
            continue
        try:
            probe = load_checkpoint_compat(candidate)
            a = probe.get('args', {}) or {}
        except Exception:
            continue
        motion = bool(a['use_motion_features']) if 'use_motion_features' in a else not bool(a.get('disable_motion_features', False))
        if (a.get('backbone', 'transformer') == 'transformer' and int(a.get('num_classes', -1)) == EXPECTED_NUM_CLASSES
                and not motion and str(a.get('pooling', 'mean')) in {'mean', 'masked_mean'}):
            found.append((candidate, a.get('d_model'), a.get('layers'), a.get('heads'), a.get('pooling', 'mean'), probe.get('epoch')))
    for row in found:
        print(row)
    if len(found) != 1:
        raise RuntimeError(f'Có {len(found)} checkpoint tương thích; gán IPN_NEURAL_BASELINE_CHECKPOINT thủ công ở cell cấu hình (khuyến nghị ablation C: d_model=192, 4 layer, 8 head, masked_mean).')
    IPN_NEURAL_BASELINE_CHECKPOINT = found[0][0]

IPN_NEURAL_BASELINE_CHECKPOINT = checkpoint_path(IPN_NEURAL_BASELINE_CHECKPOINT, 'IPN_NEURAL_BASELINE_CHECKPOINT')
baseline_payload = load_checkpoint_compat(IPN_NEURAL_BASELINE_CHECKPOINT)
BASELINE_ARGS = dict(baseline_payload.get('args', {}))
if not BASELINE_ARGS:
    raise ValueError('Baseline checkpoint thiếu args; không thể chứng minh kiến trúc tương thích.')
BASELINE_BACKBONE = BASELINE_ARGS.get('backbone', 'transformer')
BASELINE_NUM_CLASSES = int(BASELINE_ARGS.get('num_classes', -1))
BASELINE_NUM_FRAMES = int(BASELINE_ARGS.get('num_frames', 64))
BASELINE_D_MODEL = int(BASELINE_ARGS.get('d_model', -1))
BASELINE_LAYERS = int(BASELINE_ARGS.get('layers', -1))
BASELINE_HEADS = int(BASELINE_ARGS.get('heads', -1))
BASELINE_DROPOUT = float(BASELINE_ARGS.get('dropout', 0.1))
BASELINE_POOLING = str(BASELINE_ARGS.get('pooling', 'mean'))
if 'use_motion_features' in BASELINE_ARGS:
    BASELINE_MOTION_FEATURES = bool(BASELINE_ARGS['use_motion_features'])
else:
    BASELINE_MOTION_FEATURES = not bool(BASELINE_ARGS.get('disable_motion_features', False))

assert BASELINE_BACKBONE == 'transformer', f'V4 neural adapter chỉ hỗ trợ transformer, nhận {BASELINE_BACKBONE!r}'
assert BASELINE_NUM_CLASSES == EXPECTED_NUM_CLASSES, f'Baseline phải có 13 logits, nhận {BASELINE_NUM_CLASSES}'
assert BASELINE_NUM_FRAMES > 0 and BASELINE_D_MODEL > 0 and BASELINE_LAYERS > 0 and BASELINE_HEADS > 0
assert BASELINE_D_MODEL % BASELINE_HEADS == 0, 'd_model phải chia hết cho heads.'
print('Neural baseline (C) dùng velocity:', BASELINE_MOTION_FEATURES, '- chỉ dùng để đánh giá neural-only độc lập; V4 dùng checkpoint no-motion ở bước 4b.')
assert BASELINE_POOLING in {'mean', 'masked_mean'}, f'Pooling không tương thích với V4 adapter: {BASELINE_POOLING}'
if BASELINE_POOLING == 'masked_mean':
    print('CẢNH BÁO: baseline dùng masked_mean; V4 neural memory adapter dùng mean pooling. Đây là khác biệt semantics, không dùng V4 để thay baseline.')

baseline_state = {key.removeprefix('module.'): value for key, value in baseline_payload.get('model', baseline_payload).items()}
input_weight = baseline_state.get('input_proj.0.weight')
classifier_weight = baseline_state.get('classifier.4.weight')
assert input_weight is not None and tuple(input_weight.shape)[1] == 52 * (5 if BASELINE_MOTION_FEATURES else 3), 'input_proj không khớp metadata motion.'
assert classifier_weight is not None and tuple(classifier_weight.shape)[0] == EXPECTED_NUM_CLASSES

BASELINE_CACHE_ROOT = Path(IPN_NEURAL_CACHE_ROOT).expanduser() if IPN_NEURAL_CACHE_ROOT else None
if BASELINE_CACHE_ROOT is None:
    saved_cache = BASELINE_ARGS.get('cache_root')
    saved_cache_path = Path(saved_cache).expanduser() if saved_cache else None
    candidates = [saved_cache_path, DRIVE_DATA_ROOT / 'keypoint_tensor_cache', DRIVE_DATA_ROOT / 'data' / 'keypoint_tensor_cache']
    BASELINE_CACHE_ROOT = next((path.resolve() for path in candidates if path is not None and path.is_dir()), None)
if BASELINE_CACHE_ROOT is None:
    raise FileNotFoundError('Neural-only evaluator cần keypoint tensor cache; gán IPN_NEURAL_CACHE_ROOT trên Drive.')
print({'baseline': str(IPN_NEURAL_BASELINE_CHECKPOINT), 'd_model': BASELINE_D_MODEL, 'layers': BASELINE_LAYERS, 'heads': BASELINE_HEADS, 'dropout': BASELINE_DROPOUT, 'num_frames': BASELINE_NUM_FRAMES, 'pooling': BASELINE_POOLING, 'cache_root': str(BASELINE_CACHE_ROOT)})

## 4b. Checkpoint neural khởi tạo V4 (no-motion)

V4 adapter không có projection cho velocity, nên không nạp được C. Bước này train neural-only **cùng siêu tham số C** nhưng thêm `--disable-motion-features`, chỉ trên train/validation, chọn `best.pt` theo validation macro-F1. Nếu checkpoint đã tồn tại trên Drive thì dùng lại. Neural-only C (có velocity) vẫn được đánh giá riêng ở bước test như baseline mạnh nhất; checkpoint no-motion cũng được đánh giá để so sánh công bằng với V4.

In [ ]:
TRAIN_NEURAL_SCRIPT = REPO_DIR / 'final' / 'code' / 'neural_only_experiments' / 'train_neural_only.py'
assert TRAIN_NEURAL_SCRIPT.is_file(), TRAIN_NEURAL_SCRIPT
NEURAL_CWD = REPO_DIR / 'final' / 'code' / 'neural_only_experiments'
IPN_V4_INIT_CHECKPOINT = Path(IPN_V4_INIT_CHECKPOINT)
if not EVAL_ONLY and not IPN_V4_INIT_CHECKPOINT.is_file():
    init_dir = IPN_V4_INIT_CHECKPOINT.parent
    if init_dir.exists() and any(init_dir.iterdir()):
        raise FileExistsError(f'{init_dir} không rỗng nhưng thiếu best.pt; xử lý thủ công, không overwrite.')
    init_command = [
        sys.executable, '-u', TRAIN_NEURAL_SCRIPT,
        '--backbone', 'transformer',
        '--cache-root', BASELINE_CACHE_ROOT,
        '--train-file', DRIVE_MANIFESTS['train'],
        '--val-file', DRIVE_MANIFESTS['val'],
        '--output-dir', init_dir,
        '--num-classes', EXPECTED_NUM_CLASSES,
        '--num-frames', BASELINE_NUM_FRAMES,
        '--batch-size', BASELINE_ARGS.get('batch_size', 64),
        '--epochs', BASELINE_ARGS.get('epochs', 80),
        '--d-model', BASELINE_D_MODEL,
        '--layers', BASELINE_LAYERS,
        '--heads', BASELINE_HEADS,
        '--dropout', BASELINE_DROPOUT,
        '--label-smoothing', BASELINE_ARGS.get('label_smoothing', 0.05),
        '--pooling', BASELINE_POOLING,
        '--class-weight-power', BASELINE_ARGS.get('class_weight_power', 0.5),
        '--jitter-std', BASELINE_ARGS.get('jitter_std', 0.005),
        '--frame-mask-probability', BASELINE_ARGS.get('frame_mask_probability', 0.0),
        '--keypoint-dropout-probability', BASELINE_ARGS.get('keypoint_dropout_probability', 0.02),
        '--select-metric', 'macro_f1',
        '--early-stop-patience', BASELINE_ARGS.get('early_stop_patience', 12),
        '--seed', 42,
        '--device', 'cuda:0' if torch.cuda.is_available() else 'cpu',
        '--num-workers', 0,
        '--log-every', 20,
        '--disable-motion-features',
    ]
    assert '--test-file' not in init_command
    run_argv(init_command, cwd=NEURAL_CWD)
if EVAL_ONLY:
    V4_INIT_CHECKPOINT = None
else:
    V4_INIT_CHECKPOINT = checkpoint_path(IPN_V4_INIT_CHECKPOINT, 'IPN_V4_INIT_CHECKPOINT')
    init_payload = load_checkpoint_compat(V4_INIT_CHECKPOINT)
    init_args = dict(init_payload.get('args', {}))
    init_motion = bool(init_args['use_motion_features']) if 'use_motion_features' in init_args else not bool(init_args.get('disable_motion_features', False))
    assert not init_motion, 'Checkpoint khởi tạo V4 phải no-motion.'
    assert int(init_args.get('num_classes', -1)) == EXPECTED_NUM_CLASSES
    assert int(init_args.get('d_model', -1)) == BASELINE_D_MODEL and int(init_args.get('layers', -1)) == BASELINE_LAYERS and int(init_args.get('heads', -1)) == BASELINE_HEADS
    init_state = {k.removeprefix('module.'): v for k, v in init_payload.get('model', init_payload).items()}
    assert tuple(init_state['input_proj.0.weight'].shape)[1] == 52 * 3
    print('V4 init checkpoint:', V4_INIT_CHECKPOINT, 'sha256=', sha256_file(V4_INIT_CHECKPOINT))


## 5. Chỉ dựng test graph còn thiếu

Lệnh precompute có --splits test duy nhất. Các ngưỡng phase/predicate lấy trực tiếp từ run_config.json của artifact; không thay bằng matcher hay ngưỡng tùy ý.

In [ ]:
BUILD_SPLITS = ['test']
assert set(BUILD_SPLITS) == {'test'}
assert len(graph_files('train')) == EXPECTED_COUNTS['train']
assert len(graph_files('val')) == EXPECTED_COUNTS['val']
test_graphs_before = len(graph_files('test'))
assert test_graphs_before <= EXPECTED_COUNTS['test']
event_config = dict(run_config['event_config'])
predicate_thresholds = dict(run_config['predicate_thresholds'])
required_event_keys = ('max_phase_duration', 'min_phase_duration', 'min_overlap_ratio', 'min_overlap_frames')
assert all(key in event_config for key in required_event_keys)
required_predicate_keys = ('hand', 'finger', 'pinch', 'trend_consistency')
assert all(key in predicate_thresholds for key in required_predicate_keys)

missing_test_graphs = EXPECTED_COUNTS['test'] - test_graphs_before
if missing_test_graphs:
    precompute_command = [
        sys.executable, '-u', PRECOMPUTE_SCRIPT,
        '--keypoint-root', KEYPOINT_ROOT,
        '--output-root', GRAPH_ROOT,
        '--num-frames', BASELINE_NUM_FRAMES,
        '--train-file', TRAIN_FILE,
        '--val-file', VAL_FILE,
        '--test-file', TEST_FILE,
        '--splits', 'test',
        '--max-phase-duration', event_config['max_phase_duration'],
        '--min-phase-duration', event_config['min_phase_duration'],
        '--min-overlap-ratio', event_config['min_overlap_ratio'],
        '--min-overlap-frames', event_config['min_overlap_frames'],
        '--hand-motion-threshold', predicate_thresholds['hand'],
        '--finger-motion-threshold', predicate_thresholds['finger'],
        '--pinch-threshold', predicate_thresholds['pinch'],
        '--trend-consistency', predicate_thresholds['trend_consistency'],
    ]
    run_argv(precompute_command, cwd=REPO_DIR)
else:
    print('Test graph đã đủ; không chạy lại precompute.')

assert len(graph_files('train')) == EXPECTED_COUNTS['train']
assert len(graph_files('val')) == EXPECTED_COUNTS['val']
assert len(graph_files('test')) == EXPECTED_COUNTS['test']
print({'test_graphs_before': test_graphs_before, 'test_graphs_missing': missing_test_graphs, 'test_graphs_after': len(graph_files('test')), 'event_config': event_config, 'predicate_thresholds': predicate_thresholds})

## 6. Train V4 final hoặc dùng checkpoint V4 có sẵn

Train command cố ý **không có --test-file** và không có --save-best-test; best.pt chỉ được ghi khi validation macro_f1 cải thiện. Ở EVAL_ONLY, checkpoint V4 được kiểm tra là 13 lớp nhưng vẫn phải đi qua bước test-graph ở trên.

In [ ]:
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
if EVAL_ONLY:
    V4_BEST = checkpoint_path(IPN_V4_CHECKPOINT, 'IPN_V4_CHECKPOINT')
    v4_probe = load_checkpoint_compat(V4_BEST)
    v4_args = dict(v4_probe.get('args', {}))
    assert int(v4_args.get('num_classes', -1)) == EXPECTED_NUM_CLASSES, 'V4 checkpoint không phải 13 lớp.'
    print('EVAL_ONLY: dùng V4 checkpoint', V4_BEST)
else:
    train_command = [
        sys.executable, '-u', V4_TRAIN_SCRIPT,
        '--keypoint-root', KEYPOINT_ROOT,
        '--instance-graph-root', GRAPH_ROOT,
        '--template-dir', TEMPLATE_ROOT,
        '--neural-checkpoint', V4_INIT_CHECKPOINT,
        '--train-file', TRAIN_FILE,
        '--val-file', VAL_FILE,
        '--output-dir', V4_OUTPUT_DIR,
        '--logit-branch', 'final',
        '--num-classes', '13',
        '--num-frames', BASELINE_NUM_FRAMES,
        '--d-model', BASELINE_D_MODEL,
        '--layers', BASELINE_LAYERS,
        '--heads', BASELINE_HEADS,
        '--dropout', BASELINE_DROPOUT,
        '--batch-size', 32,
        '--epochs', 50,
        '--select-metric', 'macro_f1',
        '--frame-mask-probability', 0,
        '--keypoint-dropout-probability', 0.02,
        '--device', DEVICE,
        '--seed', 42,
    ]
    assert '--test-file' not in train_command
    assert '--save-best-test' not in train_command
    run_argv(train_command, cwd=REPO_DIR)
    V4_BEST = V4_OUTPUT_DIR / 'best.pt'
    assert V4_BEST.is_file(), 'V4 không tạo best.pt theo validation.'

V4_BEST = Path(V4_BEST).resolve()
V4_PAYLOAD = load_checkpoint_compat(V4_BEST)
assert int(V4_PAYLOAD.get('args', {}).get('num_classes', -1)) == EXPECTED_NUM_CLASSES
assert V4_PAYLOAD.get('args', {}).get('select_metric') == 'macro_f1' or EVAL_ONLY
print('V4 checkpoint:', V4_BEST, 'sha256=', sha256_file(V4_BEST))

## 7. Locked test — mỗi evaluator đúng một lần

Giữ RUN_LOCKED_TEST=False trong authoring/tuning. Khi gate được bật, test_cross_attention_v2.py --eval-branch final xuất đồng thời ba nhánh neural, kg, final; kg ở đây vẫn là learned template-guided head dùng neural + graph memories. Neural-only dùng evaluator độc lập và checkpoint baseline riêng.

In [ ]:
LOCAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
V4_METRICS_PATH = EVAL_OUTPUT_DIR / 'v4_test_metrics.json'
NEURAL_METRICS_PATH = EVAL_OUTPUT_DIR / 'neural_test_metrics.json'
NEURAL_NOMOTION_METRICS_PATH = EVAL_OUTPUT_DIR / 'neural_nomotion_test_metrics.json'

def optional_cache_args(command):
    return command + (['--cache-root', str(BASELINE_CACHE_ROOT)] if BASELINE_CACHE_ROOT else [])

if RUN_LOCKED_TEST:
    assert len(graph_files('test')) == EXPECTED_COUNTS['test']
    v4_test_command = [
        sys.executable, '-u', V4_TEST_SCRIPT,
        '--checkpoint', V4_BEST,
        '--keypoint-root', KEYPOINT_ROOT,
        '--instance-graph-root', GRAPH_ROOT,
        '--template-dir', TEMPLATE_ROOT,
        '--test-file', TEST_FILE,
        '--split', 'test',
        '--batch-size', 32,
        '--num-workers', 0,
        '--device', DEVICE,
        '--timing-warmup-batches', 2,
        '--timing-repeats', 1,
        '--eval-branch', 'final',
        '--output', V4_METRICS_PATH,
    ]
    v4_test_command = optional_cache_args(v4_test_command)
    run_argv(v4_test_command, cwd=REPO_DIR)
    def run_neural_eval(checkpoint, metrics_path, predictions_copy):
        command = [
            sys.executable, '-u', NEURAL_TEST_SCRIPT,
            '--checkpoint', checkpoint,
            '--cache-root', BASELINE_CACHE_ROOT,
            '--test-file', DRIVE_MANIFESTS['test'],
            '--split', 'test',
            '--batch-size', 128,
            '--num-workers', 0,
            '--device', DEVICE,
            '--timing-warmup-batches', 2,
            '--timing-repeats', 1,
            '--output', metrics_path,
        ]
        run_argv(command, cwd=REPO_DIR / 'final' / 'code' / 'neural_only_experiments')
        generated = Path(metrics_path).with_name('predictions.csv')
        assert generated.is_file()
        shutil.copy2(generated, predictions_copy)
        generated.unlink()
    run_neural_eval(IPN_NEURAL_BASELINE_CHECKPOINT, NEURAL_METRICS_PATH, EVAL_OUTPUT_DIR / 'neural_test_predictions.csv')
    if V4_INIT_CHECKPOINT is not None:
        run_neural_eval(V4_INIT_CHECKPOINT, NEURAL_NOMOTION_METRICS_PATH, EVAL_OUTPUT_DIR / 'neural_nomotion_test_predictions.csv')
    assert V4_METRICS_PATH.is_file() and NEURAL_METRICS_PATH.is_file()
    assert (EVAL_OUTPUT_DIR / 'v4_test_predictions.csv').is_file()
    assert (EVAL_OUTPUT_DIR / 'v4_test_per_class.csv').is_file()
    assert (EVAL_OUTPUT_DIR / 'v4_test_confusion_matrices.json').is_file()
else:
    print('RUN_LOCKED_TEST=False: chưa đọc test metrics và chưa chạy evaluator.')

## 8. So sánh metric, per-class và confusion

Không suy ra macro-F1 từ accuracy. Bảng giữ riêng neural-only độc lập và ba nhánh cùng forward của V4; confusion matrix có hướng true→pred và phải đủ 13×13.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def validate_matrix(matrix):
    assert len(matrix) == EXPECTED_NUM_CLASSES
    assert all(len(row) == EXPECTED_NUM_CLASSES for row in matrix)

if RUN_LOCKED_TEST:
    v4_metrics = read_json(V4_METRICS_PATH)
    neural_metrics = read_json(NEURAL_METRICS_PATH)
    assert set(v4_metrics.get('branches', {})) >= {'neural', 'kg', 'final'}
    branch_rows = []
    for branch in ('neural', 'kg', 'final'):
        summary = v4_metrics['branches'][branch]
        branch_rows.append({'model': f'V4 {branch} (learned)', 'accuracy': float(summary['accuracy']), 'macro_f1': float(summary['macro_f1'])})
    branch_rows.append({'model': 'Neural-only C (velocity, independent)', 'accuracy': float(neural_metrics['accuracy']), 'macro_f1': float(neural_metrics['macro_f1'])})
    if NEURAL_NOMOTION_METRICS_PATH.is_file():
        nomotion_metrics = read_json(NEURAL_NOMOTION_METRICS_PATH)
        branch_rows.append({'model': 'Neural-only no-motion (V4 init, independent)', 'accuracy': float(nomotion_metrics['accuracy']), 'macro_f1': float(nomotion_metrics['macro_f1'])})
    comparison_metrics = {'status': 'locked_test_complete', 'rows': branch_rows, 'v4_eval_branch': v4_metrics.get('eval_branch'), 'note': 'V4 kg_logits uses neural memory and graph memory; neural-only is a separate checkpoint.'}
    (LOCAL_RESULTS_DIR / 'comparison_metrics.json').write_text(json.dumps(comparison_metrics, ensure_ascii=False, indent=2), encoding='utf-8')

    v4_per_class = list(csv.DictReader((EVAL_OUTPUT_DIR / 'v4_test_per_class.csv').open(encoding='utf-8', newline='')))
    neural_per_class = neural_metrics.get('per_class', [])
    assert len(v4_per_class) == EXPECTED_NUM_CLASSES and len(neural_per_class) == EXPECTED_NUM_CLASSES
    per_class_rows = []
    for class_id in range(EXPECTED_NUM_CLASSES):
        v4_row = v4_per_class[class_id]
        neural_row = neural_per_class[class_id]
        assert int(v4_row['class_id']) == class_id and int(neural_row['class_id']) == class_id
        per_class_rows.append({'class_id': class_id, 'class_name': v4_row.get('class_name', str(class_id)), 'neural_f1': float(neural_row['f1']), 'v4_neural_f1': float(v4_row['neural_f1']), 'v4_kg_f1': float(v4_row['kg_f1']), 'v4_final_f1': float(v4_row['final_f1']), 'kg_fixes': int(v4_row['kg_fixes']), 'kg_harms': int(v4_row['kg_harms'])})
    with (LOCAL_RESULTS_DIR / 'comparison_per_class.csv').open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(per_class_rows[0]))
        writer.writeheader()
        writer.writerows(per_class_rows)

    v4_confusions = read_json(EVAL_OUTPUT_DIR / 'v4_test_confusion_matrices.json')
    neural_confusion = neural_metrics['confusion_matrix']
    for matrix in list(v4_confusions.values()) + [neural_confusion]:
        validate_matrix(matrix)
    confusion_payload = {'neural_only': neural_confusion, 'v4': v4_confusions, 'orientation': 'true_row_to_pred_column'}
    (LOCAL_RESULTS_DIR / 'comparison_confusions.json').write_text(json.dumps(confusion_payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print(json.dumps(comparison_metrics, ensure_ascii=False, indent=2))
else:
    comparison_metrics = {'status': 'test_not_run', 'locked_test_gate': False}
    (LOCAL_RESULTS_DIR / 'comparison_metrics.json').write_text(json.dumps(comparison_metrics, indent=2), encoding='utf-8')
    print(comparison_metrics)

## 9. Đóng gói provenance vào ZIP duy nhất trên Drive

Run directory và ZIP đều có tên duy nhất. Notebook từ chối overwrite. Gói gồm V4 checkpoint, metric/prediction nếu test đã bật, per-class/confusion, hash đầu vào/checkpoint, commit, command argv và config; không copy raw keypoint, graph cache hay output notebook.

In [ ]:
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_DIR = DRIVE_RUN_ROOT / RUN_ID
DRIVE_RUN_ZIP = DRIVE_RUN_ROOT / f'{RUN_ID}.zip'
if DRIVE_RUN_DIR.exists() or DRIVE_RUN_ZIP.exists():
    raise FileExistsError(f'Refuse overwrite: {DRIVE_RUN_DIR} hoặc {DRIVE_RUN_ZIP} đã tồn tại.')
DRIVE_RUN_DIR.mkdir(parents=False, exist_ok=False)
PACKAGE_DIR = DRIVE_RUN_DIR / 'payload'
PACKAGE_DIR.mkdir(parents=False, exist_ok=False)

def copy_into(source, relative_name):
    source = Path(source)
    if not source.is_file():
        return None
    target = PACKAGE_DIR / relative_name
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, target)
    return target

def json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    return value

copy_into(V4_BEST, 'checkpoint/v4_best.pt')
if V4_INIT_CHECKPOINT is not None:
    copy_into(V4_INIT_CHECKPOINT, 'checkpoint/v4_init_no_motion_best.pt')
copy_into(ORIGINAL_MANIFESTS / 'train.txt', 'provenance/artifact_train.txt')
copy_into(ORIGINAL_MANIFESTS / 'val.txt', 'provenance/artifact_val.txt')
if not EVAL_ONLY:
    copy_into(V4_OUTPUT_DIR / 'history.json', 'checkpoint/history.json')
for filename in ('v4_test_metrics.json', 'v4_test_predictions.csv', 'v4_test_per_class.csv', 'v4_test_confusion_matrices.json', 'neural_test_metrics.json', 'neural_test_predictions.csv', 'neural_nomotion_test_metrics.json', 'neural_nomotion_test_predictions.csv'):
    copy_into(EVAL_OUTPUT_DIR / filename, f'evaluation/{filename}')
for filename in ('comparison_metrics.json', 'comparison_per_class.csv', 'comparison_confusions.json'):
    copy_into(LOCAL_RESULTS_DIR / filename, f'evaluation/{filename}')

hashes = {
    'train_val_artifact_source': str(TRAIN_VAL_ARTIFACT_ZIP),
    'train_val_artifact_sha256': sha256_file(TRAIN_VAL_ARTIFACT_ZIP) if TRAIN_VAL_ARTIFACT_ZIP.is_file() else sha256_file(RUN_CONFIG_PATH),
    'v4_best_checkpoint': sha256_file(V4_BEST),
    'neural_baseline_checkpoint': sha256_file(IPN_NEURAL_BASELINE_CHECKPOINT),
    'v4_init_no_motion_checkpoint': sha256_file(V4_INIT_CHECKPOINT) if V4_INIT_CHECKPOINT is not None else None,
    'repo_commit': REPO_COMMIT_RESOLVED,
    'drive_manifest_sha256': {split: sha256_file(path) for split, path in DRIVE_MANIFESTS.items()},
    'normalized_manifest_sha256': MANIFEST_SHA256,
}
(PACKAGE_DIR / 'hashes.json').write_text(json.dumps(hashes, indent=2), encoding='utf-8')
(PACKAGE_DIR / 'source_artifact_run_config.json').write_text(RUN_CONFIG_PATH.read_text(encoding='utf-8'), encoding='utf-8')
(PACKAGE_DIR / 'source_artifact_completed_run.json').write_text(COMPLETED_RUN_PATH.read_text(encoding='utf-8'), encoding='utf-8')
(PACKAGE_DIR / 'code_revision.txt').write_text(REPO_COMMIT_RESOLVED + '\n', encoding='utf-8')
(PACKAGE_DIR / 'commands.json').write_text(json.dumps(COMMANDS, indent=2), encoding='utf-8')
packaged_config = dict(run_config)
packaged_config.update({
    'run_id': RUN_ID,
    'eval_only': EVAL_ONLY,
    'locked_test': RUN_LOCKED_TEST,
    'baseline_args': json_safe(BASELINE_ARGS),
    'resolved_model': {
        'd_model': BASELINE_D_MODEL,
        'layers': BASELINE_LAYERS,
        'heads': BASELINE_HEADS,
        'dropout': BASELINE_DROPOUT,
        'num_frames': BASELINE_NUM_FRAMES,
        'pooling_source': BASELINE_POOLING,
        'v4_neural_memory_pooling': 'mean',
    },
    'graph_counts': {split: len(graph_files(split)) for split in ('train', 'val', 'test')},
    'manifest_paths': {split: str(path) for split, path in DRIVE_MANIFESTS.items()},
    'keypoint_root': str(KEYPOINT_ROOT),
})
(PACKAGE_DIR / 'run_config.json').write_text(json.dumps(packaged_config, ensure_ascii=False, indent=2), encoding='utf-8')
package_manifest = {
    'run_id': RUN_ID,
    'created_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'repo_url': REPO_URL,
    'repo_commit': REPO_COMMIT_RESOLVED,
    'eval_only': EVAL_ONLY,
    'locked_test': RUN_LOCKED_TEST,
    'checkpoint': 'payload/checkpoint/v4_best.pt',
    'artifact_policy': 'No raw keypoints, graph cache, or notebook outputs embedded.',
    'hashes_file': 'payload/hashes.json',
    'commands_file': 'payload/commands.json',
    'config_file': 'payload/run_config.json',
}
(DRIVE_RUN_DIR / 'run_manifest.json').write_text(json.dumps(package_manifest, ensure_ascii=False, indent=2), encoding='utf-8')

if DRIVE_RUN_ZIP.exists():
    raise FileExistsError(f'Refuse overwrite: {DRIVE_RUN_ZIP}')
with zipfile.ZipFile(DRIVE_RUN_ZIP, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(DRIVE_RUN_DIR.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(DRIVE_RUN_DIR))
print('Run ZIP:', DRIVE_RUN_ZIP)
print('Run directory:', DRIVE_RUN_DIR)